# Download and assemble IBL behavioural data

This notebook downloads subject-level behavioural aggregates from the International Brain Laboratory dataset associated with the `2021_Q1_IBL_et_al_Behaviour` release.

For each subject, the trial, session, and training-status tables are loaded using the IBL ONE API and combined into a single trial-level dataframe. Session-level performance and training-day variables are then added, and the resulting dataset is saved as `all_trials.pqt` for use in the preprocessing notebook.

The core data-loading workflow was adapted from International Brain Laboratory and Neuromatch Academy course materials. Project-specific modifications include assembling the final combined dataframe and exporting it for the subsequent analysis pipeline.

## 1. Install dependencies and imports

In [2]:
# pip install apis
!pip install --quiet ONE-api
!pip install --quiet ibllib


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
# Imports
import os
import sys
import warnings
import numpy as np
import pandas as pd

from one.api import ONE
from one.remote.aws import s3_download_file, get_s3_public
from brainbox.behavior.training import compute_performance_easy

## 2. Connect to the IBL public database

In [4]:
# Limit download threads when running in Jupyter
os.environ.setdefault('ONE_HTTP_DL_THREADS', '1')

# Connect to the public IBL Alyx database
ONE.setup(base_url='https://openalyx.internationalbrainlab.org', silent=True)
one = ONE(password='international')

warnings.simplefilter("ignore", FutureWarning)

Connected to https://openalyx.internationalbrainlab.org as user "intbrainlab"


## 3. Define aggregate-data loading function

In [5]:
# Suppress some future warnings
warnings.simplefilter("ignore", FutureWarning)

# Change the load method depending on python version
from one.remote.aws import s3_download_file, get_s3_public
import pandas as pd
import sys
python_ver = sys.version_info

if python_ver >= (3, 10):
    from one.alf.path import add_uuid_string
else:
    from one.alf.files import add_uuid_string

s3, bucket = get_s3_public()

def load_aggregate(subject, dataset):
    if sys.version_info >= (3, 10):
        return one.load_aggregate('subjects', subject, dataset)
    else:
        files = one.list_aggregates('subjects', subject, dataset=dataset)
        files = files.iloc[0]
        src_path = str(add_uuid_string(files['rel_path'], files.name))
        dst_path = one.cache_dir.joinpath(files['rel_path'])
        local_file = s3_download_file(src_path, dst_path, s3=s3, bucket_name=bucket)
        return pd.read_parquet(local_file)

## 4. Identify subjects in the behavioural dataset

In [6]:
# Find all aggregate training datasets from IBL behaviour paper
datasets = one.alyx.rest('datasets', 'list', tag='2021_Q1_IBL_et_al_Behaviour', name='_ibl_subjectTrials.table.pqt')

# Find the subject name from the info stored in the relative path
subjects = np.unique([d['file_records'][0]['relative_path'].split('/')[2] for d in datasets])

## 5. Download and combine subject-level tables

In [7]:
# Import datasets for each subject and combine into a single dataframe
all_trials = []

# Download two tables, subject trials table and subject training table and combine
for i, subject in enumerate(subjects):
  if np.mod(i, 50) == 0:
    print(f'{i}/{len(subjects)}')

  # Load trials table for subject
  subject_trials = load_aggregate(subject, '_ibl_subjectTrials.table.pqt')

  # Load sessions table for subject
  session_trials = load_aggregate(subject, '_ibl_subjectSessions.table.pqt')

  # Load training status for subject
  subject_training = load_aggregate(subject, '_ibl_subjectTraining.table.pqt')

  # Combine the two tables
  subject_trials = (subject_trials
            .set_index('session')
            .join(subject_training.set_index('session'))
            .sort_values(by=['session_start_time', 'intervals_0']))
  subject_trials['training_status'] =  subject_trials["training_status"].ffill()

  # Join sessions table for number, task_protocol, etc.
  if 'task_protocol' in subject_trials:
      subject_trials = subject_trials.drop('task_protocol', axis=1)
  subject_trials = subject_trials.join(session_trials.drop('date', axis=1))

  # Sort the trials by session start time and trial number
  subject_trials = subject_trials.sort_values(by=['session_start_time', 'intervals_0'])


  # add in extra columns for performance easy and training day
  sessions = subject_trials.index.unique()
  for n_sess, sess in enumerate(sessions):
    trials = subject_trials[subject_trials.index == sess]
    perf_easy = compute_performance_easy(trials)
    subject_trials.loc[subject_trials.index == sess, 'performance_easy'] = perf_easy
    subject_trials.loc[subject_trials.index == sess, 'training_day'] = n_sess


  all_trials.append(subject_trials)

all_trials = pd.concat(all_trials)

0/140


/Users/namratamuralidharan/.pyenv/versions/3.12.5/lib/python3.12/site-packages/one/util.py:442: ALFWarning: No default revision for dataset aggregates/Subjects/churchlandlab/CSHL045/#2024-04-10#/_ibl_subjectTrials.table.pqt; using most recent
  warnings.warn(
/Users/namratamuralidharan/.pyenv/versions/3.12.5/lib/python3.12/site-packages/one/util.py:442: ALFWarning: No default revision for dataset aggregates/Subjects/churchlandlab/CSHL046/_ibl_subjectTrials.table.pqt; using most recent
  warnings.warn(
/Users/namratamuralidharan/.pyenv/versions/3.12.5/lib/python3.12/site-packages/one/util.py:442: ALFWarning: No default revision for dataset aggregates/Subjects/churchlandlab/CSHL047/#2024-08-21#/_ibl_subjectTrials.table.pqt; using most recent
  warnings.warn(
/Users/namratamuralidharan/.pyenv/versions/3.12.5/lib/python3.12/site-packages/one/util.py:442: ALFWarning: No default revision for dataset aggregates/Subjects/churchlandlab/CSHL049/#2024-08-21#/_ibl_subjectTrials.table.pqt; using mo

50/140


/Users/namratamuralidharan/.pyenv/versions/3.12.5/lib/python3.12/site-packages/one/util.py:442: ALFWarning: No default revision for dataset aggregates/Subjects/angelakilab/IBL-T3/_ibl_subjectTrials.table.pqt; using most recent
  warnings.warn(
/Users/namratamuralidharan/.pyenv/versions/3.12.5/lib/python3.12/site-packages/one/util.py:442: ALFWarning: No default revision for dataset aggregates/Subjects/mrsicflogellab/IBL_001/_ibl_subjectTrials.table.pqt; using most recent
  warnings.warn(
/Users/namratamuralidharan/.pyenv/versions/3.12.5/lib/python3.12/site-packages/one/util.py:442: ALFWarning: No default revision for dataset aggregates/Subjects/mrsicflogellab/IBL_002/#2024-08-21#/_ibl_subjectTrials.table.pqt; using most recent
  warnings.warn(


100/140


## 6. Save the assembled dataset

In [8]:
# Download the dataset into a parquet file
all_trials.to_parquet('all_trials.pqt')